# Μάθημα 11 - Πρωτόκολλο Πράκτορα προς Πράκτορα (A2A)


## Εγκατάσταση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Τι είναι το Πρωτόκολλο A2A;

Το **Πρωτόκολλο Πράκτορα προς Πράκτορα (A2A)** είναι ένα ανοιχτό πρότυπο που επιτρέπει στους πράκτορες AI να επικοινωνούν, να ανακαλύπτουν ο ένας τον άλλον και να συνεργάζονται — ακόμα και όταν είναι χτισμένοι σε διαφορετικά πλαίσια ή φιλοξενούνται από διαφορετικές υπηρεσίες.

Key concepts:

- **Ανακάλυψη** – Οι πράκτορες δημοσιεύουν μια *Κάρτα Πράκτορα* που περιγράφει τις δυνατότητές τους, διευκολύνοντας τους άλλους πράκτορες (ή τους ορχηστρωτές) να βρουν τον κατάλληλο ειδικό για μια εργασία.
- **Διαβίβαση Μηνυμάτων** – Οι πράκτορες ανταλλάσσουν δομημένα μηνύματα μέσω ενός κοινού πρωτοκόλλου, έτσι ώστε ένα αίτημα από έναν πράκτορα να μπορεί να γίνει κατανοητό και να εκπληρωθεί από έναν άλλο ανεξάρτητα από την εσωτερική του υλοποίηση.
- **Κύκλος Ζωής Εργασίας** – Το A2A ορίζει καταστάσεις όπως *υποβλήθηκε*, *σε εξέλιξη*, *ολοκληρώθηκε* και *απέτυχε*, παρέχοντας στον ορχηστρωτή πλήρη ορατότητα στο πώς προχωρά μια ανατεθείσα εργασία.

Σε αυτό το μάθημα προσομοιώνουμε συνεργασία με το στυλ A2A συνδέοντας τρεις εξειδικευμένους ταξιδιωτικούς πράκτορες σε μια ροή εργασίας όπου κάθε πράκτορας συνεισφέρει την τεχνογνωσία του και μεταβιβάζει τα αποτελέσματα στον επόμενο.


## Δημιουργία Εξειδικευμένων Ταξιδιωτικών Πρακτόρων


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Συνεργασία Πολλών Πρακτόρων μέσω Ροής Εργασίας

Συνδέουμε τους τρεις πράκτορες σε μια διαδοχική ροή εργασίας που αντικατοπτρίζει τη μεταβίβαση μηνυμάτων A2A:

1. **CurrencyExchangeAgent** receives the user request and produces currency guidance.
2. **ActivityPlannerAgent** receives the enriched context and adds activity recommendations.
3. **TravelManagerAgent** synthesizes both inputs into a final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Κατανόηση του A2A σε περιβάλλον παραγωγής

Σε περιβάλλον παραγωγής το πρωτόκολλο A2A απελευθερώνει ισχυρά σενάρια δια-υπηρεσιακής συνεργασίας:

| Capability | Description |
|---|---|
| **Cross-framework interop** | Ένας agent που έχει δημιουργηθεί με ένα πλαίσιο μπορεί να αναθέτει εργασίες σε έναν agent που έχει δημιουργηθεί με οποιοδήποτε άλλο πλαίσιο συμβατό με A2A, επιτρέποντας πραγματική διαλειτουργικότητα μεταξύ οργανισμών. |
| **Service boundaries** | Agents μπορούν να βρίσκονται σε ξεχωριστά microservices, cloud regions, ή ακόμα και σε διαφορετικούς οργανισμούς ενώ εξακολουθούν να συνεργάζονται αδιάλειπτα. |
| **Dynamic discovery** | Ένας orchestrator μπορεί να ερωτά ένα μητρώο Agent Card κατά τη διάρκεια εκτέλεσης για να βρει τον πιο κατάλληλο ειδικό για μια δεδομένη υποεργασία. |
| **Streaming & push notifications** | Το A2A υποστηρίζει Server-Sent Events (SSE) για ενημερώσεις προόδου σε πραγματικό χρόνο και push ειδοποιήσεις για εργασίες μεγάλης διάρκειας. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Περίληψη

Σε αυτό το μάθημα μάθατε:

1. **Τι είναι το πρωτόκολλο A2A** — ένα ανοιχτό πρότυπο για την ανακάλυψη μεταξύ πρακτόρων, την ανταλλαγή μηνυμάτων,
   και τη διαχείριση εργασιών.
2. **Πώς να δημιουργήσετε εξειδικευμένους πράκτορες** — έναν πράκτορα ανταλλαγής νομισμάτων, έναν πράκτορα προγραμματιστή δραστηριοτήτων,
   και έναν ορχηστρωτή Travel Manager.
3. **Πώς να ενσωματώσετε πράκτορες σε μια ροή εργασίας** — χρησιμοποιώντας `WorkflowBuilder` για να μοντελοποιήσετε διαδοχική
   ανταλλαγή μηνυμάτων μεταξύ πρακτόρων.
4. **Πώς λειτουργεί το A2A στην παραγωγή** — επιτρέποντας συνεργασία μεταξύ διαφορετικών πλαισίων και υπηρεσιών
   με δυναμική ανακάλυψη και ροή ενημερώσεων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Αποποίηση ευθυνών:
Το παρόν έγγραφο έχει μεταφραστεί με χρήση της υπηρεσίας αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στην αρχική του γλώσσα πρέπει να θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
